# Tutorial: the whole pipeline on a small 1KGP panel

This notebook trains Diet Network models and predicts with them, end to end, on
`kgp_mini`: 480 samples (20 from each of the 24 populations) × 2,000 SNPs of
public 1000 Genomes data, committed in `tests/fixtures/`. Every step is a
`dietnet` command, run here from Python so the files between them can be
inspected. It takes under a minute on a laptop CPU and needs no network.

With 2,000 SNPs and about 290 training samples per fold the models are far
weaker than the published ones (229,986 SNPs); the point is the pipeline, not
the accuracy.

It runs from a clone of the repository: `uv sync --frozen --extra docs`, then
open `docs/tutorial.ipynb`.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

# The fixtures are in the repository, next to docs/.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tests" / "fixtures").is_dir())
FIXTURES = REPO / "tests" / "fixtures"
PANEL = FIXTURES / "kgp_mini"
LABELS = FIXTURES / "kgp_mini_labels.tsv"

# Everything this notebook writes goes to a scratch directory.
os.chdir(tempfile.mkdtemp(prefix="dietnet-tutorial-"))


def dietnet(*args, tail=None):
    """Run a dietnet command; print its output (only the last `tail` lines if given)."""
    result = subprocess.run(
        [sys.executable, "-m", "dietnet.cli", *map(str, args)],
        capture_output=True,
        text=True,
    )
    here = os.path.realpath(os.getcwd())
    output = (result.stdout + result.stderr).replace(here + os.sep, "")
    lines = [line for line in output.splitlines() if line.strip(" -")]
    print("\n".join(lines[-tail:] if tail else lines))
    if result.returncode != 0:
        raise RuntimeError(f"dietnet {args[0]} exited with status {result.returncode}")

In [ ]:
dietnet("info")

## The inputs

A PLINK fileset and a label table: a header row, then sample id and label.

In [ ]:
labels = pd.read_csv(LABELS, sep="\t")
print(labels.head(3), "\n")
print(labels.iloc[:, 1].value_counts().describe()[["count", "min", "max"]])

In [ ]:
pd.read_csv(f"{PANEL}.bim", sep="\t", header=None,
            names=["chrom", "snp", "cm", "pos", "a1", "a2"]).head(3)

## 1. The HDF5 dataset

`create-dataset` reads the `.bed` directly and writes the HDF5 file upstream's
scripts read. Dosages count the `.bim` A1 allele, and SNP names carry it
(`<id>_<A1>`), as `plink --recode A` would name them.

In [ ]:
dietnet("create-dataset", "--plink", PANEL, "--labels", LABELS, "--out", "dataset.h5")

with h5py.File("dataset.h5") as f:
    for key in ("inputs", "samples", "snp_names", "class_labels", "class_label_names"):
        print(f"{key:18s} {f[key].dtype}  {f[key].shape}")
    print("first SNP names:", np.array(f["snp_names"][:2]).astype(str))

## 2. Folds

Upstream's `partition_data.py`: five folds, each with train, validation and
test splits, class labels balanced across them.

In [ ]:
dietnet("partition", "--dataset", "dataset.h5", "--out", "partition.npz", "--seed", 3, tail=1)

folds = np.load("partition.npz", allow_pickle=True)["folds_indexes"]
pd.DataFrame([[len(s) for s in fold] for fold in folds],
             columns=["train", "valid", "test"]).rename_axis("fold")

## 3. Per-fold inputs

The auxiliary network's input (genotype frequencies per population, from each
fold's training samples) and each SNP's mean and SD, both from upstream's
scripts.

In [ ]:
dietnet("generate-embedding", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--out", "embedding.npz", tail=1)
dietnet("compute-input-stats", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--out", "stats.npz", tail=1)

print("emb:", np.load("embedding.npz")["emb"].shape, "(folds x SNPs x 3 genotypes x 24 populations)")
stats = np.load("stats.npz")
print({key: stats[key].shape for key in stats.files})

## 4. Training

Upstream's `train.py`, once per (seed, fold). This config is small enough to
train in seconds; the published one is in the [training guide](../training/).
Two folds of one seed give a two-model ensemble.

In [ ]:
Path("config.yaml").write_text("""\
batch_size: 32
epochs: 60
input_dropout: 0.5
dropout_main: 0.5
lr_aux: 0.001
lr_main: 0.001
learning_rate_annealing: 0.99
nb_hidden_u_aux: [64, 64]
nb_hidden_u_main: [64]
patience: 60
seed: 1
uniform_init_limit: 0.02
""")

dietnet("train", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--embedding", "embedding.npz", "--input-stats", "stats.npz",
        "--config", "config.yaml", "--exp-dir", "exp", "--seeds", 1, "--folds", "0,1")
sorted(str(p) for p in Path("exp/packages").rglob("*") if p.is_file())

Each package is self-contained, and records how it was trained:

In [ ]:
from dietnet import package

packages = package.find("exp/packages")
{k: v for k, v in packages[0].metadata.items() if k not in ("classes", "results_dir")}

Each model's accuracy on its own held-out test fold (chance is 1/24):

In [ ]:
dietnet("check", "--packages", "exp/packages")

## 5. Prediction

`predict` matches the cohort to the models' SNPs by position and allele, then
scores it with every model and combines them. Here the "cohort" is the panel
itself, so every SNP is found; on another cohort the counts show how many were
copied, flipped or missing.

In [ ]:
dietnet("predict", "--model", "exp/packages", "--plink", PANEL, "--out", "predictions.tsv",
        "--device", "cpu")
predictions = pd.read_csv("predictions.tsv", sep="\t")
predictions.head()

In [ ]:
dietnet("check", "--predictions", "predictions.tsv", "--labels", LABELS, tail=1)

Most of these samples were in the models' training splits, so this
accuracy flatters them; the held-out numbers above are the honest ones.

The same through the Python API, which also gives every model's scores:

In [ ]:
from dietnet import predict

matched = predict.matched_dataset(packages, PANEL, "work")
result = predict.predict(packages, matched, device="cpu")
print("scores:", result.scores.shape, "(models x samples x classes)")
print("scale:", result.scale)
result.alignment

## 6. Other SNP features and the MLP baseline

`build-embedding` assembles the auxiliary network's input from blocks. Here:
the published features plus a block of random columns, the control for "more
columns" rather than "more information". See [SNP features](../features/).

In [ ]:
Path("features.yaml").write_text("""\
blocks:
  - provider: class_genotype_freq
  - provider: random
    columns: 16
""")
dietnet("build-embedding", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--spec", "features.yaml", "--out", "embedding_plus_random.npz")

Upstream's MLP has no auxiliary network, so it takes no embedding. It is the
baseline any change to the SNP features should beat.

In [ ]:
dietnet("train", "--model", "mlp", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--input-stats", "stats.npz", "--config", "config.yaml", "--exp-dir", "exp_mlp",
        "--seeds", 1, "--folds", "0,1")
dietnet("check", "--packages", "exp_mlp/packages")